In [35]:
import yaml
import json
import logging
import joblib
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor
from scipy.stats import randint, uniform

# Настройки отображения
pd.set_option('display.max_columns', None)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Пути
CONFIGS_DIR = Path('../configs').resolve()
ARTIFACTS_ROOT = Path('../artifacts').resolve()
MODELS_DIR = ARTIFACTS_ROOT / 'models'
METRICS_DIR = ARTIFACTS_ROOT / 'metrics'

for p in [MODELS_DIR, METRICS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Загрузка конфигов
with open(CONFIGS_DIR / 'data.yaml', 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)
with open(CONFIGS_DIR / 'experiment.yaml', 'r', encoding='utf-8') as f:
    exp_cfg = yaml.safe_load(f)

logger.info('Конфигурации загружены')

2026-05-24 22:51:22,004 - INFO - Конфигурации загружены


In [ ]:
df = pd.read_csv(
    data_cfg['source_path'],
    sep=data_cfg['separator'],
    decimal=data_cfg['decimal']
)
df = df.dropna(how=data_cfg['dropna_how']).reset_index(drop=True)

num_cols = data_cfg['columns']['numeric']
cat_cols = data_cfg['columns']['categorical']
target_cols = data_cfg['columns']['targets']  # ['Y1', 'Y2']

df[cat_cols] = df[cat_cols].astype('category')

X = df.drop(columns=target_cols)
y = df[target_cols].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=exp_cfg['random_state']
)

y_train = pd.DataFrame(y_train, columns=target_cols)
y_test = pd.DataFrame(y_test, columns=target_cols)

logger.info(f'Data split: train={len(X_train)}, test={len(X_test)}')
print(f"Фактические колонки y_train: {list(y_train.columns)}")

2026-05-24 22:51:22,048 - INFO - Data split: train=614, test=154


📊 Фактические колонки y_train: ['Y1', 'Y2']


In [37]:
# Препроцессор
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# Базовые модели для Stacking
base_models = [
    ('rf', RandomForestRegressor(random_state=exp_cfg['random_state'], n_jobs=-1)),
    ('cb', CatBoostRegressor(verbose=0, random_state=exp_cfg['random_state'], allow_writing_files=False))
]

# Создаем Stacking Regressor
stacking_reg = StackingRegressor(
    estimators=base_models,
    final_estimator=Ridge(),
    cv=3,
    n_jobs=1  # n_jobs=1 внутри Stacking, чтобы не конфликтовать с Parallel
)

# Полный пайплайн
pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', stacking_reg)
])

param_distributions = {
    'model__rf__n_estimators': randint(50, 200),
    'model__rf__max_depth': [None, 10, 20],
    'model__cb__learning_rate': uniform(0.01, 0.2),
    'model__cb__depth': randint(4, 10),
    'model__final_estimator__alpha': uniform(0.01, 10.0)
}

logger.info('Pipeline and param grid defined')
print(f"Параметры для поиска: {list(param_distributions.keys())}")

2026-05-24 22:51:22,082 - INFO - Pipeline and param grid defined


Параметры для поиска: ['model__rf__n_estimators', 'model__rf__max_depth', 'model__cb__learning_rate', 'model__cb__depth', 'model__final_estimator__alpha']


In [38]:
target = 'Y2'
#  Позиционное извлечение (игнорирует имена колонок)
col_idx = target_cols.index(target)
y_train_target = y_train.iloc[:, col_idx].to_numpy().ravel()

logger.info(f'Starting tuning for {target}...')

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=exp_cfg['random_state'],
    n_jobs=1,
    return_train_score=False,
    error_score='raise'
)

search.fit(X_train, y_train_target)

logger.info(f'Best params for {target}: {search.best_params_}')
logger.info(f'Best R2 (CV): {search.best_score_:.4f}')

best_pipeline_y2 = search.best_estimator_
joblib.dump(best_pipeline_y2, MODELS_DIR / f'StackingTuned_{target}.pkl')
logger.info(f'Model saved: StackingTuned_{target}.pkl')

2026-05-24 22:51:22,104 - INFO - Starting tuning for Y2...


Fitting 3 folds for each of 10 candidates, totalling 30 fits


2026-05-24 22:52:58,631 - INFO - Best params for Y2: {'model__cb__depth': 6, 'model__cb__learning_rate': np.float64(0.19186408041575642), 'model__final_estimator__alpha': np.float64(2.597799816000169), 'model__rf__max_depth': 10, 'model__rf__n_estimators': 183}
2026-05-24 22:52:58,632 - INFO - Best R2 (CV): 0.9891
2026-05-24 22:52:58,694 - INFO - Model saved: StackingTuned_Y2.pkl


In [39]:
target = 'Y1'
col_idx = target_cols.index(target)
y_train_target = y_train.iloc[:, col_idx].to_numpy().ravel()

logger.info(f'Starting tuning for {target}...')

search_y1 = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=exp_cfg['random_state'],
    n_jobs=1,
    return_train_score=False
)

search_y1.fit(X_train, y_train_target)

logger.info(f'Best params for {target}: {search_y1.best_params_}')
logger.info(f'Best R2 (CV): {search_y1.best_score_:.4f}')

best_pipeline_y1 = search_y1.best_estimator_
joblib.dump(best_pipeline_y1, MODELS_DIR / f'StackingTuned_{target}.pkl')
logger.info(f'Model saved: StackingTuned_{target}.pkl')

2026-05-24 22:52:58,710 - INFO - Starting tuning for Y1...


Fitting 3 folds for each of 10 candidates, totalling 30 fits


2026-05-24 22:54:47,564 - INFO - Best params for Y1: {'model__cb__depth': 5, 'model__cb__learning_rate': np.float64(0.04636499344142013), 'model__final_estimator__alpha': np.float64(1.8440450985343382), 'model__rf__max_depth': 10, 'model__rf__n_estimators': 71}
2026-05-24 22:54:47,565 - INFO - Best R2 (CV): 0.9982
2026-05-24 22:54:47,621 - INFO - Model saved: StackingTuned_Y1.pkl


In [40]:
manifest_path = MODELS_DIR / 'best_model_manifest.json'

# Загружаем старый манифест (или создаем новый)
if manifest_path.exists():
    with open(manifest_path, 'r', encoding='utf-8') as f:
        manifest = json.load(f)
else:
    manifest = {}

# Обновляем данные
manifest['model_name'] = 'StackingTuned'
manifest['targets'] = target_cols
manifest['artifact_files'] = [f'StackingTuned_{t}.pkl' for t in target_cols]
manifest['timestamp'] = datetime.now().isoformat()
manifest['selection_criteria'] = 'RandomizedSearchCV on Stacking (Ridge final)'
manifest['tuning_details'] = {
    'n_iter': 10,
    'cv': 3,
    'best_params_Y2': search.best_params_,
    'best_score_Y2': float(search.best_score_),
    'best_params_Y1': search_y1.best_params_,
    'best_score_Y1': float(search_y1.best_score_)
}

# Сохраняем обновленный манифест
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

logger.info(f'Manifest updated: {manifest_path}')
print("Манифест успешно обновлен! API теперь будет использовать StackingTuned.")

2026-05-24 22:54:47,681 - INFO - Manifest updated: C:\ИИИ\AIE\project\artifacts\models\best_model_manifest.json


Манифест успешно обновлен! API теперь будет использовать StackingTuned.


In [41]:
# Проверяем качество новой модели на отложенном тесте
test_scores = {}
for target in target_cols:
    model_path = MODELS_DIR / f'StackingTuned_{target}.pkl'
    model = joblib.load(model_path)
    y_pred = model.predict(X_test)
    score = r2_score(y_test[target], y_pred)
    test_scores[target] = round(score, 4)
    logger.info(f'Test R2 for {target}: {score:.4f}')

print(f"\nИтоговые метрики на тесте (StackingTuned): {test_scores}")

2026-05-24 22:54:47,881 - INFO - Test R2 for Y1: 0.9988
2026-05-24 22:54:48,219 - INFO - Test R2 for Y2: 0.9875



Итоговые метрики на тесте (StackingTuned): {'Y1': 0.9988, 'Y2': 0.9875}
